# 02 — RFM 用户画像与聚类分析

**分析目标**：基于用户历史交互记录，计算 RFM（近度/频度/价值）三维指标，通过 KMeans 聚类将用户划分为不同群体，为差异化推荐策略提供依据。

**RFM 定义**：
- **R (Recency)**：用户距参考日期最近一次交互的天数，越小表示越活跃
- **F (Frequency)**：用户历史总交互次数
- **M (Monetary)**：用户累计消费价值，以 `sum(rating × price)` 近似

In [ ]:
import sys
sys.path.insert(0, "../src")

from pathlib import Path
import polars as pl
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib
matplotlib.rcParams['font.family'] = ['SimHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

DATA_DIR = Path("../data")
INTERIM = DATA_DIR / "interim"
FIGURES = Path("../reports/figures")
FIGURES.mkdir(parents=True, exist_ok=True)

# 加载数据
df = pl.read_parquet(INTERIM / "interactions.parquet")
print(f"已加载数据：{len(df):,} 条交互记录，{df['user_id'].n_unique():,} 个用户")
print(f"字段列表：{df.columns}")

## 1. RFM 指标计算

In [ ]:
from ecom_rec.features.rfm import compute_rfm, REFERENCE_DATE_UNIX

rfm = compute_rfm(df, reference_ts=REFERENCE_DATE_UNIX)

print("RFM 基本统计：")
print(rfm.select(["recency_days", "frequency", "monetary"]).describe())

print(f"\nRFM 样例（前 5 行）：")
print(rfm.head(5))

## 2. 肘部法则选择最优 K

In [ ]:
from ecom_rec.features.profile import find_optimal_k

k_range = range(2, 9)
inertias = find_optimal_k(rfm, k_range=k_range)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=list(k_range),
    y=inertias,
    mode="lines+markers",
    marker=dict(size=8, color="#2196F3"),
    line=dict(color="#2196F3", width=2)
))
fig.update_layout(
    title="肘部法则选择最优聚类数 K",
    xaxis_title="K 值",
    yaxis_title="Inertia（总内离差平方和）",
    template="plotly_white",
    width=700,
    height=400
)
fig.write_image(str(FIGURES / "09_elbow.png"), scale=2)
fig.show()

print("各 K 值对应 Inertia：")
for k, inertia in zip(k_range, inertias):
    print(f"  K={k}: {inertia:.2f}")

## 3. KMeans 聚类（k=4）

In [ ]:
from ecom_rec.features.profile import cluster_users

rfm_labeled = cluster_users(rfm, n_clusters=4, random_state=42)

print("各用户群体统计：")
segment_summary = (
    rfm_labeled.group_by("user_segment")
    .agg([
        pl.len().alias("用户数"),
        pl.col("recency_days").mean().round(1).alias("平均近度(天)"),
        pl.col("frequency").mean().round(1).alias("平均频度"),
        pl.col("monetary").mean().round(2).alias("平均价值"),
    ])
    .sort("平均价值", descending=True)
)
print(segment_summary)

## 4. 用户分布可视化

In [ ]:
# 3D RFM 散点图
rfm_pd = rfm_labeled.to_pandas()

fig = px.scatter_3d(
    rfm_pd,
    x="recency_days",
    y="frequency",
    z="monetary",
    color="user_segment",
    title="RFM 用户分群 3D 可视化",
    labels={
        "recency_days": "近度(天)",
        "frequency": "频度",
        "monetary": "价值"
    },
    opacity=0.6,
    color_discrete_map={
        "核心高价值用户": "#2196F3",
        "潜力用户": "#4CAF50",
        "沉睡用户": "#FF9800",
        "流失用户": "#F44336",
    },
    template="plotly_white"
)
fig.write_image(str(FIGURES / "10_rfm_3d_scatter.png"), scale=2)
fig.show()

In [ ]:
# 各群体特征雷达图（归一化 RFM 值）
from sklearn.preprocessing import MinMaxScaler

segment_stats = rfm_labeled.group_by("user_segment").agg([
    pl.col("recency_days").mean().alias("recency_days"),
    pl.col("frequency").mean().alias("frequency"),
    pl.col("monetary").mean().alias("monetary"),
]).to_pandas()

scaler = MinMaxScaler()
cols = ["recency_days", "frequency", "monetary"]
segment_stats_norm = segment_stats.copy()
segment_stats_norm[cols] = scaler.fit_transform(segment_stats[cols])
# Recency 反转（越小越好 → 越大越好）
segment_stats_norm["recency_days"] = 1 - segment_stats_norm["recency_days"]

categories = ["近度(反转)", "频度", "价值"]
fig = go.Figure()
colors = ["#2196F3", "#4CAF50", "#FF9800", "#F44336"]
for i, row in segment_stats_norm.iterrows():
    values = [row["recency_days"], row["frequency"], row["monetary"]]
    fig.add_trace(go.Scatterpolar(
        r=values + [values[0]],
        theta=categories + [categories[0]],
        fill="toself",
        name=row["user_segment"],
        line_color=colors[i % len(colors)],
        opacity=0.7,
    ))
fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    title="各用户群体 RFM 特征雷达图（归一化）",
    template="plotly_white",
    width=700,
    height=500
)
fig.write_image(str(FIGURES / "11_rfm_radar.png"), scale=2)
fig.show()

In [ ]:
# 各群体用户数量条形图
seg_count = (
    rfm_labeled.group_by("user_segment")
    .agg(pl.len().alias("user_count"))
    .sort("user_count", descending=True)
    .to_pandas()
)

color_map = {
    "核心高价值用户": "#2196F3",
    "潜力用户": "#4CAF50",
    "沉睡用户": "#FF9800",
    "流失用户": "#F44336",
}
bar_colors = [color_map.get(s, "#9E9E9E") for s in seg_count["user_segment"]]

fig = go.Figure(data=[
    go.Bar(
        x=seg_count["user_segment"],
        y=seg_count["user_count"],
        marker_color=bar_colors,
        text=[f"{c:,}" for c in seg_count["user_count"]],
        textposition="outside",
    )
])
fig.update_layout(
    title="各用户群体规模分布",
    xaxis_title="用户群体",
    yaxis_title="用户数量",
    template="plotly_white",
    width=700,
    height=450,
    showlegend=False
)
fig.write_image(str(FIGURES / "12_segment_distribution.png"), scale=2)
fig.show()

## 5. 用户群体解读

基于 KMeans 聚类（k=4）将用户划分为以下四类：

| 群体 | 标签 | 特征描述 | 推荐策略 |
|------|------|----------|----------|
| 核心高价值用户 | High Value | R 小（活跃）、F 高（频繁）、M 高（高消费） | 精品推荐、会员权益、个性化新品 |
| 潜力用户 | Potential | R 中等、F 中等、M 中等，有上升趋势 | 提频促活、满减券、关联商品推荐 |
| 沉睡用户 | Dormant | R 大（较久未访问）、F/M 中等 | 唤醒营销、限时折扣、推送个性化召回 |
| 流失用户 | Churned | R 最大（长期未访问）、F/M 低 | 低成本触达、大力度促销或放弃维护 |

**关键结论**：
- 核心高价值用户虽然数量少，但贡献了大部分 Monetary 价值，推荐系统应优先保证该群体体验
- 潜力用户是增长重点，通过精准推荐提升其活跃度和购买频次
- 沉睡/流失用户的召回成本较高，需结合业务ROI决策是否投入资源